In [ ]:
import glob
import json
import re
import warnings
from pathlib import Path

import astropy.units as un
import h5py
import matplotlib.pyplot as plt
import numpy as np
import py21cmfast as p21c
from astropy.cosmology import Planck18 as cosmo
from matplotlib import colormaps, colors, rcParams
from py21cmfast.drivers._global_initialization import c_state
from py21cmfast.io.caching import OutputCache, RunCache
from scipy.interpolate import RegularGridInterpolator, interp1d

warnings.filterwarnings("ignore", category=DeprecationWarning)
rcParams.update({"font.size": 20, "font.family": "serif"})

eor_colour = colors.LinearSegmentedColormap.from_list(
    "EoR",
    [
        (0, "white"),
        (0.21, "yellow"),
        (0.42, "orange"),
        (0.63, "red"),
        (0.86, "black"),
        (0.9, "blue"),
        (1, "cyan"),
    ],
)
try:
    colormaps.register(cmap=eor_colour)
except ValueError:
    pass
print("21cmFAST", p21c.__version__)

# Table of Contents
Configuration:
0. [Models and paths](#config)

Comparison with Ivan's results (galaxy model; should not change between EOS models):
1. [Escape fraction](#fesc)
2. [SHMR](#SHMR)
3. [SFMS](#SFMS)
4. [Mh-Muv](#Mh-Muv)
5. [HMF](#hmf)
6. [UV LFs](#uvlfs)

Reionization (changes between EOS models):

7. [EoR history](#xhi)
8. [Thomson scattering optical depth](#tau)
9. [Lightcones](#lcs)
10. [Global quantities](#globals)

# Models and paths <a class="anchor" id="config"></a>
Everything this notebook reads is in the repository (run it from `postprocess/`) or in the
simulation caches listed in `MODELS`. Ivan's curves and `interpolation_table.npy` (K_UV) are in
`postprocess/`; the observed UV LFs (Bouwens+21, Donnan+24, Chemerynska+26) are the 21CMMC data
files, copied to `postprocess/UVLF_data/`.

Each entry of `MODELS`: `template` (TOML), `overrides` (as the run scripts pass them, e.g. the
test `HII_DIM`), `cache` (21cmFAST cache directory), `xhi` (global history: a lightcone `.h5`, a
glob of coeval logs, or a `.json` of global means), `grids` (`"cache"` to read the feedback grids
from the cache) and `max_halos` (read at most this many halos per redshift; `None` = all).

In [ ]:
import sys

HERE = Path.cwd()  # postprocess/
REPO = HERE.parent
sys.path.insert(0, str(REPO / "run_scripts"))
import settings  # noqa: E402  (DATA_ROOT: $EOS26_DATA if set, /g/data/qp00/$USER on Gadi, else the repository root)

DATA = HERE  # Ivan's txt files and interpolation_table.npy (K_UV)
UVLF_DATA = HERE / "UVLF_data"  # observed UV LFs, copied from 21CMMC (src/py21cmmc/data)
PLOTS = REPO / "plots" / "summaries"
DATA_ROOT = Path(settings.DATA_ROOT)
EARLY_LOGS = REPO / "logs" / "EOS26_earlyEoR" / "EOS26_coeval_*.log"  # EOS26_earlyEoR production coeval logs


def smallbox_model(template, color):
    """The newest EOS26_smallbox_xHI.ipynb run of `template` with its own f_esc, or None.

    Its cache (DATA_ROOT/EOS26_smallbox_dim<N>_cell<L>_cache) holds the full simulation, and its history
    is the xHI_history_*.json written next to it. Caches of older versions of that notebook have no coevals.
    """
    ap = p21c.InputParameters.from_template(str(template)).astro_params
    fesc = {k: getattr(ap, k) for k in ("F_ESC10_ACG", "ALPHA_ESC", "F_ESC7_MCG")}
    runs = []
    for fn in DATA_ROOT.glob("EOS26_smallbox_dim*_cache/xHI_history_*.json"):
        d = json.load(open(fn))
        if Path(d.get("template", "")).name == Path(template).name and all(
                abs(d["fesc"][k] - v) < 1e-9 for k, v in fesc.items()):
            runs.append((fn.stat().st_mtime, fn, d))
    if not runs:
        return None
    _, fn, d = max(runs)
    overrides = {k: d[k] for k in ("HII_DIM", "LOWRES_CELL_SIZE_MPC", "N_THREADS", "random_seed")}
    return dict(template=template, overrides=overrides, cache=fn.parent, xhi=fn, grids="cache", max_halos=None, color=color)


# TEST = True: the EOS26 test box (HII_DIM = 200 at 1.5 cMpc, 300 cMpc) if its halo catalogues are in
#              DATA_ROOT/EOS26_test_HIIDIM200; otherwise the small box of EOS26_smallbox_xHI.ipynb
#              (if its cache has no coevals: no feedback UV LFs and no global Tb/Ts). Both are compared with the
#              EOS26_earlyEoR production history from its coeval logs (its test boxes were deleted).
# TEST = False: the production runs, EOS26 and EOS26_earlyEoR (halos read up to max_halos per redshift).
TEST = True
if TEST:
    test_cache = DATA_ROOT / "EOS26_test_HIIDIM200"
    test_overrides = dict(HII_DIM=settings.TEST_HII_DIM, random_seed=settings.TEST_RANDOM_SEED)
    test_inputs = p21c.InputParameters.from_template(str(REPO / "EOS26.toml"), **test_overrides)
    test_rc = RunCache.from_inputs(test_inputs, cache=OutputCache(str(test_cache)))
    small = None
    if not any(Path(p).exists() for p in test_rc.HaloCatalog.values()):
        small = smallbox_model(REPO / "EOS26.toml", color="C0")
    if small is None:
        EOS26_LABEL = "EOS26"
        MODELS = {EOS26_LABEL: dict(template=REPO / "EOS26.toml", overrides=test_overrides, cache=test_cache,
                                    xhi=settings.lightcone_path(test_cache), grids="cache", max_halos=None, color="C0")}
    else:
        o = small["overrides"]
        EOS26_LABEL = f"EOS26 small box {o['HII_DIM']}^3"
        MODELS = {EOS26_LABEL: small}
        print(f"No EOS26 test halos in {test_cache}: using the small box in {small['cache']}")
    EXTRA_HISTORIES = {"EOS26_earlyEoR production (coeval logs)": dict(src=EARLY_LOGS, color="k")}
    XHI_PLOT = ["EOS26_earlyEoR production (coeval logs)", EOS26_LABEL]
else:
    # 1500^3: read at most 2e8 halos per redshift (a slab of the box) for the galaxy plots
    EOS26_LABEL = "EOS26"
    MODELS = {
        "EOS26": dict(template=REPO / "EOS26.toml", overrides={}, cache=DATA_ROOT / "EOS26_cache",
                      xhi=REPO / "logs" / "EOS26_coeval_*.log", grids="cache", max_halos=2e8, color="C0"),
        "EOS26_earlyEoR": dict(template=REPO / "EOS26_earlyEoR.toml", overrides={}, cache=DATA_ROOT / "EOS26_earlyEoR_cache",
                               xhi=EARLY_LOGS, grids="cache", max_halos=2e8, color="k"),
    }
    EXTRA_HISTORIES = {}
    XHI_PLOT = ["EOS26_earlyEoR", "EOS26"]
UVLF_REF = EOS26_LABEL  # denominator of the UV LF ratio panels

In [ ]:
def model_inputs(m):
    return p21c.InputParameters.from_template(str(m["template"]), **m.get("overrides", {}))


missing = []
for label, m in MODELS.items():
    m["inputs"] = model_inputs(m)
    m["runcache"] = RunCache.from_inputs(m["inputs"], cache=OutputCache(str(m["cache"])))
    so, ap = m["inputs"].simulation_options, m["inputs"].astro_params
    n_halo = sum(Path(p).exists() for p in m["runcache"].HaloCatalog.values())
    n_coeval = sum(Path(p).exists() for p in m["runcache"].IonizedBox.values())
    print(
        f"{label:10s} HII_DIM={so.HII_DIM} cell={so.BOX_LEN / so.HII_DIM:.2f} Mpc BOX_LEN={so.BOX_LEN:.0f} Mpc  "
        f"F_ESC10_ACG={ap.F_ESC10_ACG:+.3f} ALPHA_ESC={ap.ALPHA_ESC:+.3f} F_ESC7_MCG={ap.F_ESC7_MCG:+.3f}  "
        f"in cache: {n_halo} halo catalogues, {n_coeval} coevals of {len(m['inputs'].node_redshifts)}"
    )
    if n_halo == 0:
        ics = m["runcache"].InitialConditions
        missing.append(f"{label}: {m['cache']} (e.g. {next(iter(ics.values())) if isinstance(ics, dict) else ics})")
if missing:
    raise FileNotFoundError(
        f"No halo catalogues for these inputs (21cmFAST {p21c.__version__}) in\n  " + "\n  ".join(missing)
        + "\nWith TEST = True, run pbs_scripts/full_test_job.sh or EOS26_smallbox_xHI.ipynb first. The caches are found below settings.DATA_ROOT "
        "(/g/data/qp00/$USER on Gadi, the repository root elsewhere, or $EOS26_DATA). "
        "A cache written by 21cmFAST 4.2 has other paths and is not found by 4.3."
    )
REF = list(MODELS)[0]
UVLF_REF = globals().get("UVLF_REF", REF)  # denominator of the UV LF ratios
PLOTS.mkdir(parents=True, exist_ok=True)
TAG = re.sub(r"[^A-Za-z0-9.]+", "_", "_vs_".join(MODELS)).strip("_")


def colour(i, label):
    """A model's plot colour: MODELS[label]["color"] if given, else the i-th default colour."""
    return MODELS[label].get("color", f"C{i}") if label in MODELS else f"C{i}"

In [ ]:
# ---- reading halos, feedback grids and global histories --------------------------------
HALO_FIELDS = ["halo_masses", "halo_coords", "star_rng", "sfr_rng", "xray_rng"]


def _existing(d):
    """The boxes of a RunCache listing that are on disk."""
    return {k: v for k, v in d.items() if Path(v).exists()}


def _closest(d, z, within=0.1):
    d = _existing(d)
    if not d:
        raise KeyError(f"no box of this kind on disk (z={z})")
    zs = np.array(sorted(d))
    zc = zs[np.argmin(np.abs(zs - z))]
    if abs(zc - z) > within:
        raise KeyError(f"no box within {within} of z={z} (closest {zc:.3f})")
    return zc


def read_halos(m, z, within=0.1):
    """Halo catalogue closest to z, read with h5py (at most m['max_halos'] halos).

    Returns (z_node, dict of arrays, volume in Mpc^3 that the halos read represent).
    """
    zc = _closest(m["runcache"].HaloCatalog, z, within)
    with h5py.File(m["runcache"].HaloCatalog[zc], "r") as f:
        g = f["HaloCatalog/OutputFields"]
        n = g["halo_masses"].shape[0]
        k = n if m.get("max_halos") is None else min(n, int(m["max_halos"]))
        h = {key: g[key][:k] for key in HALO_FIELDS}
    vol = m["inputs"].simulation_options.BOX_LEN ** 3 * (k / max(n, 1))
    return zc, h, vol


def _read_field(path, struct, field):
    with h5py.File(path, "r") as f:
        return f[f"{struct}/OutputFields/{field}"][...]


def read_vcb(m):
    """The low-res relative-velocity field of the model's ICs."""
    if m.get("grids", "cache") == "cache":
        ics = m["runcache"].InitialConditions
        ics = next(iter(ics.values())) if isinstance(ics, dict) else ics
        return _read_field(ics, "InitialConditions", "lowres_vcb")
    return np.load(Path(m["grids"]) / "vcb.npy")


def feedback_grids(m, z):
    """vcb, J_21_LW, z_reion, Gamma12 seen by halos at node z.

    As in the simulation, the reionization and LW feedback at a node come from the
    previous (higher-z) node's IonizedBox and TsBox.
    """
    rc = m["runcache"]
    if m.get("grids", "cache") == "cache":
        vcb = read_vcb(m)
        zs = np.array(sorted(_existing(rc.IonizedBox)))
        higher = zs[zs > z + 1e-6]
        if len(higher) == 0:
            raise KeyError(f"no IonizedBox above z={z:.3f} in {m['cache']} for the feedback grids")
        zp = higher.min()
        zre = _read_field(rc.IonizedBox[zp], "IonizedBox", "z_reion")
        g12 = _read_field(rc.IonizedBox[zp], "IonizedBox", "ionisation_rate_G12")
        j21 = _read_field(rc.TsBox[_closest(rc.TsBox, zp, 1e-3)], "TsBox", "J_21_LW")
        return vcb, j21, zre, g12
    # test runs: grids saved by the small-box runner at the node itself
    gd = Path(m["grids"])
    f = sorted(gd.glob("grids_z*.npz"), key=lambda p: abs(float(p.stem[7:]) - z))[0]
    g = np.load(f)
    return np.load(gd / "vcb.npy"), g["J21_LW"], g["z_reion"], g["Gamma12"]


def halo_properties(m, z, feedback=False, within=0.1):
    """Galaxy properties of the halos at the node closest to z (21cmFAST's own scaling relations).

    feedback=False: no LW or reionization feedback (the galaxy model alone).
    feedback=True: with the model's own J_21_LW, z_reion and Gamma12 grids.
    """
    zc, h, vol = read_halos(m, z, within)
    inp = m["inputs"]
    lo = (inp.simulation_options.HII_DIM,) * 3
    if feedback:
        vcb, j21, zre, g12 = feedback_grids(m, zc)
    else:
        vcb = read_vcb(m)
        j21, zre, g12 = np.zeros(lo), -np.ones(lo), np.zeros(lo)
    # PR 7 (#826): convert_halo_properties needs the sigma(M) tables set up
    with c_state(inp, ps=True, sigma=True):
        p = p21c.wrapper.cfuncs.convert_halo_properties(
            redshift=zc, inputs=inp, halo_masses=h["halo_masses"], star_rng=h["star_rng"],
            sfr_rng=h["sfr_rng"], xray_rng=h["xray_rng"], halo_coords=h["halo_coords"],
            vcb_grid=vcb, J_21_LW_grid=j21, z_re_grid=zre, Gamma12_grid=g12,
        )
    p["redshift"], p["volume"] = zc, vol
    return p


def xhi_from_coeval_logs(pattern):
    """Mean x_HI at each node from the DEBUG argument summaries of the production coeval logs."""
    res = {}
    for fn in sorted(glob.glob(str(pattern))):
        lines = open(fn, errors="replace").read().splitlines()
        for i, line in enumerate(lines):
            if "Calling ComputeBrightnessTemp with following args" in line:
                z = float(lines[i + 1].split()[-1])
                for j in range(i + 2, min(i + 200, len(lines))):
                    if "neutral_fraction:" in lines[j]:
                        vals = re.findall(r"[-+]?\d\.\d+e[-+]\d+", lines[j])
                        if len(vals) == 5:
                            res[round(z, 4)] = float(vals[4])
                        break
    z = np.array(sorted(res))
    return z, np.array([res[k] for k in z])


def load_xhi(src):
    """(z ascending, mean x_HI) from a lightcone .h5, a coeval-log glob or a .json of global means."""
    src = str(src)
    if src.endswith(".h5"):
        lc = p21c.LightCone.from_file(src)
        z, x = np.array(lc.node_redshifts), np.array(lc.global_quantities["neutral_fraction"])
    elif src.endswith(".json"):
        rows = json.load(open(src))["rows"]
        z, x = np.array([r["z"] for r in rows]), np.array([r["xHI"] for r in rows])
    else:
        z, x = xhi_from_coeval_logs(src)
    o = np.argsort(z)
    return z[o], x[o]


def z_at_xhi(z, x, level):
    """Redshift at which x_HI first drops below level (coming from high z), interpolated."""
    o = np.argsort(z)[::-1]
    zs, xs = z[o], x[o]
    i = np.argmax(xs < level)
    if xs[i] >= level:
        return np.nan
    return np.interp(level, [xs[i], xs[i - 1]], [zs[i], zs[i - 1]])

In [ ]:
# ---- analytic relations (unchanged) -----------------------------------------------------
def get_M_ast(f_ast, alpha_ast, M_h, M=10):
    if not hasattr(f_ast, "__len__"):
        f_ast = np.array([f_ast])
    if not hasattr(alpha_ast, "__len__"):
        alpha_ast = np.array([alpha_ast])
    M = 10**(M)#M_sun
    m_ast = np.zeros((alpha_ast.shape[0], M_h.shape[0]))
    for i in range(M_h.shape[0]):
        m_ast[:,i] = 10**f_ast * (10**M_h[i] / M)**alpha_ast * cosmo.Ob0/cosmo.Om0 * 10**M_h[i]
    return  np.log10(m_ast).squeeze()

def get_M_ast_v4(f_ast, alpha_ast, M_h, M=10, M_pivot=14.44, alpha_ast_2=-0.61, M_turn=5.0):
    if hasattr(f_ast, "__len__"):
        raise ValueError("Only one param at a time!")
    if hasattr(alpha_ast, "__len__"):
        raise ValueError("Only one param at a time!")
    M = 10**(M)#M_sun
    M_pivot = 10**M_pivot
    M_turn = 10**M_turn
    M_h = 10**M_h
    pivot_term = ((M_pivot/M)**alpha_ast + (M_pivot/M)**alpha_ast_2) / ((M_h/M_pivot)**(-1.*alpha_ast) + (M_h/M_pivot)**(-1.*alpha_ast_2))
    return  np.log10(10**f_ast * pivot_term * np.exp(-M_turn/M_h) * cosmo.Ob0/cosmo.Om0 * M_h).squeeze()

def get_SFMS(M_ast, t_ast, z):
    if not hasattr(M_ast, "__len__"):
        M_ast = np.array([M_ast])
    if not hasattr(z, "__len__"):
        z = np.array([z])
    sfms = np.zeros(( z.shape[0],M_ast.shape[0]))
    for i in range(z.shape[0]):
        H = cosmo.H(z[i]).to(1/un.year).value
        sfms[i,:] = np.log10((10**M_ast) / (t_ast / H))
    return sfms


def f_esc(M, f10, alpha, f7, mcg=False):
    """Escape fraction of 21cmFAST (power law, capped at 1); f10, f7 in log10."""
    f = 10**f7 * (M / 1e7) ** alpha if mcg else 10**f10 * (M / 1e10) ** alpha
    return np.minimum(f, 1.0)

In [ ]:
# ---- UV magnitudes and LFs -------------------------------------------------------------
KUV_TABLE = DATA / "interpolation_table.npy"
_kuv = np.load(KUV_TABLE)
_kuv_interp = RegularGridInterpolator(
    (np.linspace(5, 15, 10), np.logspace(3, 13, 100), np.logspace(-5, 5, 100)), _kuv, method="linear"
)


def interp_kuv(SFR, Mstar, z):
    """K_UV(z, M*, SFR) from Ivan's table (SFR in Msun/yr); inputs are clipped to the table range."""
    zz = np.clip(np.ones_like(Mstar) * z, 5, 15)
    return 1 / _kuv_interp((zz, np.clip(Mstar, 1e3, 1e13), np.clip(SFR, 1e-5, 1e5)))


def Luv2Muv(Luv):
    return 51.63 - 2.5 * np.log10(Luv)


def get_muv(props, components="both"):
    """M_UV of every halo with M*_ACG > 1000 Msun and SFR > 0 (as in the original notebook)."""
    sfr = props["sfr_acg"] + (props["sfr_mcg"] if components == "both" else 0.0)
    sfr = sfr * un.yr.to(un.s)  # Msun/s -> Msun/yr
    mask = (props["stellar_mass_acg"] > 1000.0) & (sfr > 0)
    k_uv = interp_kuv(sfr[mask], props["stellar_mass_acg"][mask], props["redshift"])
    return Luv2Muv(sfr[mask] / k_uv)


UVLF_EDGES = np.arange(-24.0, -11.99, 0.5)


def get_uvlf(props, edges=UVLF_EDGES):
    muv = get_muv(props)
    hist, _ = np.histogram(muv, edges)
    centres = 0.5 * (edges[1:] + edges[:-1])
    phi = hist / (np.diff(edges) * props["volume"])
    with np.errstate(divide="ignore"):
        err = np.sqrt(hist) / (np.diff(edges) * props["volume"])
    return {"M_uv": centres, "lfunc": np.where(hist > 0, phi, np.nan), "poisson": err, "n": hist}


def load_lf(prefix, zstr):
    f = UVLF_DATA / f"{prefix}_lfuncs_z{zstr}.npz"
    if not f.exists():
        return None
    d = np.load(f)
    return d["Muv"], d["lfunc"], np.load(UVLF_DATA / f"{prefix}_sigmas_z{zstr}.npz")["sigma"]

# Escape fraction <a class="anchor" id="fesc"></a>

In [ ]:
Mh_acg, Mh_mcg = np.logspace(7, 13, 300), np.logspace(5, 10, 300)
# one curve per distinct (F_ESC10_ACG, ALPHA_ESC, F_ESC7_MCG); ACGs drawn above 1e7, MCGs below 1e10 Msun
sets = {}
for label, m in MODELS.items():
    ap = m["inputs"].astro_params
    sets.setdefault((ap.F_ESC10_ACG, ap.ALPHA_ESC, ap.F_ESC7_MCG), []).append(label)
fig, ax = plt.subplots(figsize=(10, 5))
for i, ((f10, al, f7), labels) in enumerate(sets.items()):
    name = ", ".join(labels)
    c = colour(list(MODELS).index(labels[-1]), labels[-1])
    ax.plot(Mh_acg, f_esc(Mh_acg, f10, al, f7), color=c, lw=2, label=f"{name}: ACG")
    ax.plot(Mh_mcg, f_esc(Mh_mcg, f10, al, f7, mcg=True), color=c, ls="--", lw=2, label=f"{name}: MCG")
ax.axvspan(3e7, 1e8, color="grey", alpha=0.15, lw=0)  # atomic-cooling mass (T_vir = 1e4 K) at z = 15 to 6
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(r"M$_{\rm h}$ [M$_\odot$]"); ax.set_ylabel(r"f$_{\rm esc}$")
ax.legend(fontsize=11, frameon=False)
plt.savefig(PLOTS / f"fesc_{TAG}.png", bbox_inches="tight")
plt.show()

# SHMR <a class="anchor" id="SHMR"></a>
Galaxy properties without feedback (the galaxy model alone), all node redshifts, 10,000 random halos.

In [ ]:
rng = np.random.default_rng(1)
shmr = {}
for label, m in MODELS.items():
    mh, ms = [], []
    for z in m["inputs"].node_redshifts:
        try:
            p = halo_properties(m, z)
        except KeyError:
            continue
        sel = p["halo_mass"] > 0
        mh.append(p["halo_mass"][sel]); ms.append(p["stellar_mass_acg"][sel])
    mh, ms = np.log10(np.concatenate(mh)), np.log10(np.concatenate(ms))
    idx = rng.permutation(len(mh))[:10000]
    shmr[label] = (mh[idx], ms[idx])
    print(label, len(mh), "halos")

ap = MODELS[REF]["inputs"].astro_params
M_h_ACG = np.log10(np.logspace(8, 14, 1000))
SHMR_ACG = get_M_ast_v4(ap.F_STAR10_ACG, ap.ALPHA_STAR_ACG, M_h_ACG,
                        M_pivot=ap.UPPER_STELLAR_TURNOVER_MASS,
                        alpha_ast_2=ap.UPPER_STELLAR_TURNOVER_INDEX,
                        M_turn=ap.M_TURN_STELLAR_FEEDBACK)
M_h_MCG = np.log10(np.logspace(5, 8, 1000))
SHMR_MCG = get_M_ast(ap.F_STAR7_MCG, ap.ALPHA_STAR_MCG, M_h_MCG, M=7)

# these are 1sigma CLs
M_h_Xu = np.array([6.75,7.25,7.75,8.25,8.75,9.25])
M_ast_mean_normal_Xu = np.array([3.44,3.56,4.14,5.03,6.24,7.54])
M_ast_upper_normal_Xu = np.array([0.35,0.51,0.75,0.74,0.32,0.81])
M_ast_lower_normal_Xu = np.array([0.33,0.58,0.71,0.63,0.42,0.81])
M_ast_mean_void_Xu = np.array([3.26,3.56,4.02,5.11,6.32,7.46])
M_ast_upper_void_Xu = np.array([0.22,0.49,0.61,1.10,0.74,0.12])
M_ast_lower_void_Xu = np.array([0.2,0.46,0.63,1.20,0.99,0.12])
M_ast_mean_rarepeak_Xu = np.array([3.15,3.67,4.20,5.38,6.44,7.23])
M_ast_upper_rarepeak_Xu = np.array([0.02,0.71,0.74,0.78,0.61,0.43])
M_ast_lower_rarepeak_Xu = np.array([0.02,0.81,0.72,0.78,0.45,0.51])
lower_Xu = np.array([M_ast_mean_normal_Xu - M_ast_lower_normal_Xu, M_ast_mean_void_Xu - M_ast_lower_void_Xu, M_ast_mean_rarepeak_Xu - M_ast_lower_rarepeak_Xu])
upper_Xu = np.array([M_ast_mean_normal_Xu + M_ast_upper_normal_Xu, M_ast_mean_void_Xu + M_ast_upper_void_Xu, M_ast_mean_rarepeak_Xu + M_ast_upper_rarepeak_Xu])
Xu_color = "#f4c430"

M_h_Ivan = np.log10(np.loadtxt(DATA / "halo_masses.txt"))
SHMR_Ivan = np.log10(np.loadtxt(DATA / "median_stellar_masses.txt"))

fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(9, 6), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
ax[0].plot(M_h_ACG, SHMR_ACG, label="ACG", color="lime", ls="--")
ax[0].plot(M_h_MCG, SHMR_MCG, label="MCG", color="g", ls="--")
ax[0].plot(M_h_Xu, M_ast_mean_normal_Xu, color=Xu_color, alpha=0.5)
ax[0].fill(np.append(M_h_Xu, M_h_Xu[::-1]), np.append(np.min(lower_Xu, axis=0), np.max(upper_Xu, axis=0)[::-1]), facecolor="none", edgecolor=Xu_color, linewidth=3)
ax[0].plot(M_h_Ivan, SHMR_Ivan, label="Ivan median", color="r")
mh_bins = np.linspace(9.8, 12, 21)
mh_centres = (mh_bins[1:] + mh_bins[:-1]) / 2.0
ivans = interp1d(M_h_Ivan, SHMR_Ivan, bounds_error=False)(mh_centres)
for i, (label, (mh, ms)) in enumerate(shmr.items()):
    med = np.array([np.median(ms[(mh > a) & (mh <= b)]) if np.any((mh > a) & (mh <= b)) else np.nan
                    for a, b in zip(mh_bins[:-1], mh_bins[1:])])
    ax[0].scatter(mh_centres, med, color=colour(i, label), marker="o" if i == 0 else "x", label=f"{label} median")
    ax[1].scatter(mh_centres, (ivans - med) / ivans * 100.0, color=colour(i, label), marker="o" if i == 0 else "x")
ax[1].axhline(0, color="r")
ax[1].set_xlabel(r"M$_{\rm h}$ [M$_\odot]$")
ax[0].set_ylabel(r"log$_{10}$[M$_{\ast}$/M$_\odot$]")
ax[1].set_ylabel(r"$\frac{\mathrm{Ivan - EOS}}{\mathrm{Ivan}} \times$ 100 [%]", fontsize=12)
ax[0].legend(frameon=False, fontsize=12)
plt.savefig(PLOTS / f"SHMR_{TAG}.png", bbox_inches="tight")
plt.show()

# SFMS <a class="anchor" id="SFMS"></a>

In [ ]:
M_STAR = np.log10(np.logspace(5, 12, 100))
SFMS = get_SFMS(M_STAR, ap.t_STAR, 10.0)
M_star_Ivan = np.log10(np.loadtxt(DATA / "stellar_masses_SFMS.txt"))
SFR_Ivan = np.log10(np.loadtxt(DATA / "median_SFR.txt"))

z10 = {label: halo_properties(m, 10.0) for label, m in MODELS.items()}
fig, ax = plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(10, 5), gridspec_kw={"height_ratios": [3, 1]})
ax[0].plot(M_STAR, SFMS[0, :], color="cyan", label="Analytic t*/H")
ax[0].plot(M_star_Ivan, SFR_Ivan, color="r", label="Ivan median SFR z = 10")
mast_bins = np.linspace(6, 9, 21)
mast_centres = (mast_bins[1:] + mast_bins[:-1]) / 2.0
ivans = interp1d(M_star_Ivan, SFR_Ivan, bounds_error=False)(mast_centres)
for i, (label, p) in enumerate(z10.items()):
    sel = p["sfr_acg"] > 0
    lms, lsfr = np.log10(p["stellar_mass_acg"][sel]), np.log10(p["sfr_acg"][sel] * un.yr.to(un.s))
    med = np.array([np.median(lsfr[(lms > a) & (lms <= b)]) if np.any((lms > a) & (lms <= b)) else np.nan
                    for a, b in zip(mast_bins[:-1], mast_bins[1:])])
    ax[0].scatter(mast_centres, med, color=colour(i, label), marker="o" if i == 0 else "x", label=f"{label} z = {p['redshift']:.2f} median")
    ax[1].scatter(mast_centres, (ivans - med) / ivans * 100.0, color=colour(i, label), marker="o" if i == 0 else "x")
ax[1].axhline(0, color="r")
ax[1].set_xlabel(r"log$_{10}$M$_{\ast}$ [M$_\odot$]")
ax[0].set_ylabel(r"log$_{10}$ SFR [M$_{\odot}$/yr]")
ax[1].set_ylabel(r"$\frac{\mathrm{Ivan - EOS}}{\mathrm{Ivan}} \times$ 100 [%]", fontsize=12)
ax[0].legend(fontsize=12, frameon=False, loc=2)
plt.savefig(PLOTS / f"SFR_{TAG}.png", bbox_inches="tight")
plt.show()

# Mh-Muv <a class="anchor" id="Mh-Muv"></a>

In [ ]:
M_h_Ivan_uv = np.log10(np.loadtxt(DATA / "halo_masses_Muv.txt"))
Muv_Ivan = np.loadtxt(DATA / "median_Muv_Ivan.txt")
fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(10, 5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax[0].plot(M_h_Ivan_uv, Muv_Ivan, label="Ivan median z = 10", color="r")
mh_bins = np.linspace(7, 15, 41)
mh_centres = (mh_bins[1:] + mh_bins[:-1]) / 2.0
ivans = interp1d(M_h_Ivan_uv, Muv_Ivan, bounds_error=False)(mh_centres)
for i, (label, p) in enumerate(z10.items()):
    sfr = (p["sfr_acg"] + p["sfr_mcg"]) * un.yr.to(un.s)
    sel = (p["stellar_mass_acg"] > 1000.0) & (sfr > 0)
    muv = Luv2Muv(sfr[sel] / interp_kuv(sfr[sel], p["stellar_mass_acg"][sel], p["redshift"]))
    lmh = np.log10(p["halo_mass"][sel])
    med = np.array([np.median(muv[(lmh > a) & (lmh <= b)]) if np.any((lmh > a) & (lmh <= b)) else np.nan
                    for a, b in zip(mh_bins[:-1], mh_bins[1:])])
    ax[0].scatter(mh_centres, med, color=colour(i, label), marker="o" if i == 0 else "x", label=f"{label} z = {p['redshift']:.2f} median")
    ax[1].scatter(mh_centres, (ivans - med) / ivans * 100.0, color=colour(i, label), marker="o" if i == 0 else "x")
ax[1].axhline(0, color="r")
ax[1].set_xlabel(r"log$_{10}$M$_{h}$ [M$_\odot$]")
ax[0].set_ylabel(r"M$_{UV}$")
ax[1].set_ylabel(r"$\frac{\mathrm{Ivan - EOS}}{\mathrm{Ivan}} \times$ 100 [%]", fontsize=12)
ax[0].legend(fontsize=12, loc=1, frameon=False)
plt.savefig(PLOTS / f"Mh_Muv_{TAG}.png", bbox_inches="tight")
plt.show()

# HMF <a class="anchor" id="hmf"></a>

In [ ]:
import hmf

z = 10.0
edges = np.logspace(7, 13, num=64)
dlnm = np.log(edges[1:]) - np.log(edges[:-1])
centres = (edges[:-1] * np.exp(dlnm / 2)).astype("f4")
little_h = cosmo.h
hmf_loc = hmf.MassFunction(z=z, Mmin=5, Mmax=19, dlog10m=0.05, hmf_model="SMT", transfer_params={"extrapolate_with_eh": True})
logm_to_check = np.log10(hmf_loc.m / cosmo.h)
dndlog10m_to_check = hmf_loc.dndlog10m * cosmo.h**3 * np.exp(-5e8 / (hmf_loc.m / cosmo.h))

fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(9, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax[0].loglog(hmf_loc.m / little_h, hmf_loc.dndlnm * little_h**3, color="C0", lw=2, ls=":", label="reference HMF (SMT)")
ax[0].plot(10**logm_to_check, dndlog10m_to_check / np.log(10), color="r", label="Ivan's HMF")
ref_interp = interp1d(np.log(hmf_loc.m / little_h), hmf_loc.dndlnm * little_h**3)
for i, (label, m) in enumerate(MODELS.items()):
    zc, h, vol = read_halos(m, z)
    hist, _ = np.histogram(h["halo_masses"], edges)
    mf = np.where(hist > 0, hist / vol / dlnm, np.nan)
    ax[0].plot(centres, mf, color=colour(i, label), ls="-" if i == 0 else "--", label=f"{label} z = {zc:.2f}")
    ax[1].plot(centres, mf / ref_interp(np.log(centres)) - 1, color=colour(i, label), ls="-" if i == 0 else "--")
ax[1].axhline(0, color="C0", ls=":")
ax[1].set_ylim(-0.5, 0.5)
ax[0].set_xlim([1e9, 5e12]); ax[0].set_ylim([1e-7, 1e1])
ax[0].set_ylabel("dn/dlnM (Mpc$^{-3}$)"); ax[1].set_ylabel("EOS/ref $-$ 1"); ax[1].set_xlabel("M (Msun)")
ax[0].legend(fontsize=12, frameon=False)
plt.savefig(PLOTS / f"HMF_z10_{TAG}.png", bbox_inches="tight")
plt.show()

In [ ]:
# Sampled HMF / SMT at several redshifts. Dotted lines: each model's low-res cell mass
# (rho_m * cell^3); the CHMF sampler gives an excess just below it and a deficit above.
edges = np.logspace(np.log10(5e9), 12, 26)
cen = np.sqrt(edges[1:] * edges[:-1])
dln = np.diff(np.log(edges))
rho_m = (cosmo.critical_density0 * cosmo.Om0).to("Msun/Mpc3").value
fig, axs = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for ax, z in zip(axs.flat, [6.0, 8.0, 10.0, 12.5]):
    mf = hmf.MassFunction(z=z, Mmin=9, Mmax=13, dlog10m=0.02, hmf_model="SMT", transfer_params={"extrapolate_with_eh": True})
    ref = np.interp(np.log(cen), np.log(mf.m / cosmo.h), mf.dndlnm * cosmo.h**3)
    seen = set()
    for i, (label, m) in enumerate(MODELS.items()):
        zc, h, vol = read_halos(m, z)
        n, _ = np.histogram(h["halo_masses"], edges)
        ok = n >= 5
        r = n / vol / dln / ref
        ax.errorbar(cen[ok], r[ok], yerr=(np.sqrt(n) / vol / dln / ref)[ok], color=colour(i, label), marker="o", ms=3, label=label)
        so = m["inputs"].simulation_options
        mcell = rho_m * (so.BOX_LEN / so.HII_DIM) ** 3
        if round(mcell, -8) not in seen:
            ax.axvline(mcell, color=colour(i, label), ls=":", lw=1)
            seen.add(round(mcell, -8))
    ax.axhline(1, color="k", lw=0.5)
    ax.set_xscale("log"); ax.set_ylim(0.3, 1.7); ax.set_title(f"z = {zc:.2f}", fontsize=14)
    ax.set_ylabel("dn/dlnM / SMT", fontsize=13)
for ax in axs[1]:
    ax.set_xlabel(r"M$_{\rm h}$ [M$_\odot$]")
axs[0, 0].legend(fontsize=11, frameon=False)
plt.tight_layout()
plt.savefig(PLOTS / f"HMF_ratio_{TAG}.png", bbox_inches="tight")
plt.show()

# UVLFs <a class="anchor" id="uvlfs"></a>
Solid: galaxy model without feedback. Dashed: with each model's own LW and reionization
feedback (only this part can differ between models with the same galaxy parameters).
Lower panel: ratio to the first model without feedback.

In [ ]:
uvlfs_ivan = np.loadtxt(DATA / "Ivan_UVLFs.txt")
ivan_muv = np.linspace(-23, -17, 10)
uvlf_redshifts = [6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.5, 14.5]
# 21CMMC file labels: Donnan+24 bins z = 12.5, 14.5 are stored as z12, z14
DONNAN_LABEL = {9.0: "9", 10.0: "10", 11.0: "11", 12.5: "12", 14.5: "14"}
BOUWENS_LABEL = {6.0: "6", 7.0: "7", 8.0: "8", 10.0: "10"}
CHEMERYNSKA_LABEL = {9.0: "9", 10.0: "10", 11.0: "11"}

uvlfs = {label: {} for label in MODELS}
for label, m in MODELS.items():
    for z in uvlf_redshifts:
        uvlfs[label][z] = {False: get_uvlf(halo_properties(m, z))}
        try:
            uvlfs[label][z][True] = get_uvlf(halo_properties(m, z, feedback=True))
        except KeyError as err:  # no coevals in the cache (e.g. the small-box cache): no feedback curve
            uvlfs[label][z][True] = None
            print(f"{label} z={z}: without feedback only ({err})")

def max_dev(lf, ref, sel):
    """Largest |phi/phi_ref - 1| over the selected bins with >= 10 galaxies in both."""
    if lf is None or ref is None:
        return np.nan
    ok = sel & (lf["n"] >= 10) & (ref["n"] >= 10)
    return np.nanmax(np.abs(lf["lfunc"][ok] / ref["lfunc"][ok] - 1)) if ok.any() else np.nan


centres = 0.5 * (UVLF_EDGES[1:] + UVLF_EDGES[:-1])
observed = centres < -17.0  # magnitudes covered by the HST/JWST points
uvlf_summary = []
for k, z in enumerate(uvlf_redshifts):
    fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(9, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
    for prefix, lab, mk, table in [("LF", "HST (Bouwens+21)", "s", BOUWENS_LABEL),
                                   ("Donnan24_LF", "JWST (Donnan+24)", "d", DONNAN_LABEL),
                                   ("Chemerynska26_LF", "JWST (Chemerynska+26)", "^", CHEMERYNSKA_LABEL)]:
        if z in table and (d := load_lf(prefix, table[z])) is not None:
            ax[0].errorbar(d[0], d[1], yerr=d[2], color="k", ls="none", marker=mk, label=lab)
    ax[0].plot(ivan_muv, uvlfs_ivan[k], color="r", label="Ivan's")
    for i, label in enumerate(MODELS):
        for fb, ls in [(False, "-"), (True, "--")]:
            lf, ref = uvlfs[label][z][fb], uvlfs[UVLF_REF][z][fb]
            if lf is None or ref is None:
                continue
            ax[0].plot(lf["M_uv"], lf["lfunc"], color=colour(i, label), ls=ls, label=f"{label}" + (" (feedback)" if fb else ""))
            ok = (lf["n"] >= 10) & (ref["n"] >= 10)
            ax[1].plot(lf["M_uv"][ok], lf["lfunc"][ok] / ref["lfunc"][ok], color=colour(i, label), ls=ls)
        uvlf_summary.append(dict(z=z, model=label, **{
            f"{'fb' if fb else 'nofb'}_{rng}": max_dev(uvlfs[label][z][fb], uvlfs[UVLF_REF][z][fb], sel)
            for fb in (False, True) for rng, sel in (("obs", observed), ("faint", ~observed))}))
    ax[1].axhline(1, color="k", lw=0.5)
    ax[1].set_ylim(0.8, 1.2)
    ax[1].set_ylabel(f"/ {UVLF_REF}", fontsize=11)
    ax[1].set_xlabel(r"M$_{UV}$")
    ax[0].set_ylabel(r"$\phi$ [Mpc$^{-3}$ mag$^{-1}$]")
    ax[0].set_yscale("log")
    ax[0].xaxis.set_inverted(True)
    ax[0].set_title(f"z = {z}")
    ax[0].legend(frameon=False, fontsize=10, loc=4)
    plt.savefig(PLOTS / f"UVLF_z{str(z).replace('.', 'pt')}_{TAG}.png", bbox_inches="tight")
    plt.show()

print(f"Largest |phi/phi_ref - 1| (ref: {UVLF_REF}, same feedback setting), bins with >= 10 galaxies in both;")
print("'obs' = M_UV < -17 (observed range), 'faint' = M_UV > -17")
print(f"{'z':>5s} {'model':28s} {'no fb obs':>10s} {'no fb faint':>12s} {'fb obs':>8s} {'fb faint':>9s}")
for r in uvlf_summary:
    print(f"{r['z']:5.1f} {r['model']:28s} {r['nofb_obs']:10.3f} {r['nofb_faint']:12.3f} {r['fb_obs']:8.3f} {r['fb_faint']:9.3f}")

# EoR history <a class="anchor" id="xhi"></a>

In [ ]:
Qin25_z = [5.44, 6.35, 6.75, 7.17, 7.7, 8.2, 8.5, 8.85, 9.18, 9.65, 10.01, 10.19, 10.30, 10.87, 11.88, 12.30, 13.144, 14.30]
Qin25_xHI = [0.01, 0.2, 0.3, 0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.84, 0.85, 0.86, 0.9, 0.94, 0.952, 0.96947, 0.9822]

# EXTRA_HISTORIES values: a source (path or glob) or dict(src=..., color=...).
# XHI_PLOT: the histories drawn in the x_HI and tau plots (default: all); the table lists all.
histories, hist_colour = {}, {}
for i, (label, m) in enumerate(MODELS.items()):
    if not m.get("xhi") or not glob.glob(str(m["xhi"])):
        print(f"{label}: no x_HI history at {m.get('xhi')}, skipped")
        continue
    histories[label], hist_colour[label] = load_xhi(m["xhi"]), colour(i, label)
EXTRA_COLOURS = ["k", "0.55", "C4", "C6", "C8", "C9"]
for j, (label, v) in enumerate(EXTRA_HISTORIES.items()):
    src, col = (v["src"], v.get("color")) if isinstance(v, dict) else (v, None)
    zx = load_xhi(src) if glob.glob(str(src)) else None
    if zx is None or len(zx[0]) == 0:
        print(f"{label}: no x_HI history at {src}, skipped")
        continue
    histories[label], hist_colour[label] = zx, col or EXTRA_COLOURS[j % len(EXTRA_COLOURS)]
XHI_PLOT = [label for label in (globals().get("XHI_PLOT") or list(histories)) if label in histories]

fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(12, 7), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
ax[0].scatter(Qin25_z, Qin25_xHI, color="r", label="Qin+25", zorder=5)
for label in XHI_PLOT:
    z, x = histories[label]
    ax[0].plot(z, x, color=hist_colour[label], lw=2, label=label)
    ax[1].plot(Qin25_z, np.interp(Qin25_z, z, x) - Qin25_xHI, color=hist_colour[label], lw=2)
ax[1].set_xlabel("Redshift")
ax[0].set_ylabel(r"$\overline{x}_{\rm HI}$")
ax[1].set_ylabel(r"EOS $-$ Qin25", fontsize=14)
ax[1].axhline(0, color="r")
ax[0].set_xlim(5, 18)
ax[0].legend(fontsize=11, frameon=False)
plt.savefig(PLOTS / f"xHI_{TAG}.png", bbox_inches="tight")
plt.show()

LEVELS = [0.9, 0.75, 0.5, 0.25, 0.1, 0.05, 0.01]
print(f"{'':36s} " + " ".join(f"{'x=' + str(l):>7s}" for l in LEVELS))
print(f"{'Qin+25':36s} " + " ".join(f"{np.interp(l, Qin25_xHI, Qin25_z):7.2f}" for l in LEVELS))
for label, (z, x) in histories.items():
    print(f"{label:36s} " + " ".join(f"{z_at_xhi(z, x, l):7.2f}" for l in LEVELS))

# Thomson scattering optical depth <a class="anchor" id="tau"></a>
Histories that stop above z = 5 are extended to z = 5 with their last x_HI.

In [ ]:
tau_mean, tau_sigma_u, tau_sigma_l = 0.0544, 0.0073, 0.0073  # Planck 2018 (TT,TE,EE+lowE+lensing)
taus = {}
for label, (z, x) in histories.items():
    zz, xx = (np.concatenate([[5.0], z]), np.concatenate([[x[0]], x])) if z.min() > 5.0 + 1e-6 else (z, x)
    taus[label] = p21c.compute_tau(redshifts=zz, global_xHI=xx, inputs=MODELS[REF]["inputs"])
    print(f"tau_e {label:36s} {taus[label]:.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
for label in XHI_PLOT:
    ax.axvline(taus[label], color=hist_colour[label], lw=2, label=label)
ax.axvline(tau_mean, color="k", label="Planck18")
ax.axvspan(tau_mean - tau_sigma_l, tau_mean + tau_sigma_u, color="k", alpha=0.3)
ax.set_xlabel(r"Thomson scattering optical depth $\tau_e$")
ax.legend(loc=1, fontsize=11, frameon=False)
plt.savefig(PLOTS / f"tau_{TAG}.png", bbox_inches="tight")
plt.show()

# Lightcones <a class="anchor" id="lcs"></a>
Only for models whose `xhi` is a lightcone file.

In [ ]:
keys = ["density", "brightness_temp", "cumulative_recombinations", "neutral_fraction", "spin_temperature", "tau_21", "J_21_LW"]
key_names = [r"$\delta$", r"$\delta T_{21}$ [mK]", "cumul recombs", "xHI", r"$T_S$ [K]", r"$\tau_{21}$", r"$J_{21,LW}$"]
cmaps = ["viridis", "EoR", "Reds", "hot", "inferno", "YlGnBu", "Blues"]
for label, m in MODELS.items():
    if not str(m["xhi"]).endswith(".h5") or not Path(m["xhi"]).exists():
        continue
    lightcone = p21c.LightCone.from_file(str(m["xhi"]))
    ks = [k for k in keys if k in lightcone.lightcones]
    BOX_LEN, HII_DIM = lightcone.simulation_options.BOX_LEN, lightcone.simulation_options.HII_DIM
    fig, axs = plt.subplots(nrows=1, ncols=len(ks), sharey=True, figsize=(3.5 * len(ks), 17))
    for i, k in enumerate(ks):
        c = cmaps[keys.index(k)]
        im = axs[i].pcolormesh(np.linspace(0, BOX_LEN, HII_DIM), lightcone.lightcone_redshifts, lightcone.lightcones[k][0, ...].T,
                               vmin=np.nanpercentile(lightcone.lightcones[k][0, ...], 5) if c != "EoR" else -150,
                               vmax=np.nanpercentile(lightcone.lightcones[k][0, ...], 95) if c != "EoR" else 30, cmap=c)
        plt.colorbar(im, orientation="horizontal", ax=axs[i], label=key_names[keys.index(k)], pad=0.05)
        axs[i].set_yscale("log")
    axs[0].set_ylabel("Redshifts")
    axs[0].set_title(label)
    plt.savefig(PLOTS / f"lightcones_{label}.png", bbox_inches="tight")
    plt.show()

# Global quantities <a class="anchor" id="globals"></a>

In [ ]:
keys = ["brightness_temp", "neutral_fraction", "spin_temperature"]
key_names = [r"$\delta T_{21}$ [mK]", "xHI", r"$T_S$ [K]"]
fig, axs = plt.subplots(ncols=1, nrows=3, sharex=True, figsize=(12, 9))
for i, (label, m) in enumerate(MODELS.items()):
    src = str(m["xhi"])
    if src.endswith((".h5", ".json")) and not Path(src).exists():
        continue
    if src.endswith(".h5"):
        lc = p21c.LightCone.from_file(src)
        z, g = np.array(lc.node_redshifts), {k: np.array(lc.global_quantities[k]) for k in keys}
    elif src.endswith(".json"):
        rows = json.load(open(src))["rows"]
        if not all(k in rows[0] for k in ("Tb", "Ts")):
            print(f"{label}: {Path(src).name} has x_HI only, skipped")
            continue
        z = np.array([r["z"] for r in rows])
        g = {"brightness_temp": np.array([r["Tb"] for r in rows]), "neutral_fraction": np.array([r["xHI"] for r in rows]),
             "spin_temperature": np.array([r["Ts"] for r in rows])}
    else:
        continue
    for j, k in enumerate(keys):
        axs[j].plot(z, g[k], color=colour(i, label), label=label)
for j, k_name in enumerate(key_names):
    axs[j].set_ylabel(k_name)
axs[2].set_yscale("log")
axs[0].legend(fontsize=12, frameon=False)
axs[-1].set_xlabel("Redshift")
plt.savefig(PLOTS / f"globals_{TAG}.png", bbox_inches="tight")
plt.show()